In [1]:
"""
GIBL Hackathon 2026 — Track D
Alert Triage & False Positive Reduction
-----------------------------------------
Input:  ids_alerts.csv
        incident_tickets.csv  (used for tactic enrichment)
Output: alert_triage_results.csv
 
Approach: supervised binary classification (Random Forest)
Target:   is_true_positive  (35% TP baseline, 65% FP baseline)
 
No hardcoded rules — the model learns which behavioral
patterns distinguish real attacks from noise.
"""

'\nGIBL Hackathon 2026 — Track D\nAlert Triage & False Positive Reduction\n-----------------------------------------\nInput:  ids_alerts.csv\n        incident_tickets.csv  (used for tactic enrichment)\nOutput: alert_triage_results.csv\n\nApproach: supervised binary classification (Random Forest)\nTarget:   is_true_positive  (35% TP baseline, 65% FP baseline)\n\nNo hardcoded rules — the model learns which behavioral\npatterns distinguish real attacks from noise.\n'

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

In [3]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
ALERTS_CSV   = "ids_alerts.csv"
TICKETS_CSV  = "incident_tickets.csv"
OUTPUT_CSV   = "alert_triage_results.csv"
RANDOM_STATE = 42
 
print("=" * 60)
print("  GIBL Alert Triage — Random Forest Classifier")
print("=" * 60)

  GIBL Alert Triage — Random Forest Classifier


In [4]:
# ── 1. LOAD ──────────────────────────────────────────────────────────────────
print("\n[1] Loading data...")
alerts  = pd.read_csv(ALERTS_CSV)
tickets = pd.read_csv(TICKETS_CSV)
print(f"    Alerts          : {len(alerts):,}")
print(f"    Incident tickets: {len(tickets):,}")
print(f"    Baseline TP rate: {alerts['is_true_positive'].mean()*100:.1f}%")


[1] Loading data...
    Alerts          : 150,000
    Incident tickets: 5,000
    Baseline TP rate: 35.0%


In [5]:
# ── 2. TICKET ENRICHMENT (label-free) ────────────────────────────────────────
# Extract the MITRE tactics that appear in SOC-confirmed real incidents.
# This is derived from incident_tickets, not from alert labels — no leakage.
# The model uses this as a weak prior: alerts matching confirmed-attack tactics
# may be more worth investigating.
print("\n[2] Extracting confirmed tactics from incident tickets...")
 
confirmed = tickets[tickets["is_confirmed_attack"] == True]
confirmed_tactics = set()
for chain in confirmed["tactic_chain"].dropna():
    for tactic in chain.split("|"):
        confirmed_tactics.add(tactic.strip())
 
print(f"    Confirmed incidents : {len(confirmed):,}")
print(f"    Unique tactics seen : {len(confirmed_tactics)}")


[2] Extracting confirmed tactics from incident tickets...
    Confirmed incidents : 3,829
    Unique tactics seen : 10


In [6]:
# ── 3. FEATURE ENGINEERING ───────────────────────────────────────────────────
# All features are computed purely from alert metadata — no ground-truth labels.
print("\n[3] Engineering behavioral features...")
 
df = alerts.copy()
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
df["ts_epoch"]  = df["timestamp"].astype(np.int64) // 1_000_000_000  # unix seconds
 


[3] Engineering behavioral features...


In [7]:
# -- Time features --
# Real attacks often occur at unusual times; backup/patch FPs cluster in
# predictable off-hours windows. The model learns the actual pattern.
df["hour"]         = df["timestamp"].dt.hour
df["day_of_week"]  = df["timestamp"].dt.dayofweek   # 0 = Monday
df["is_weekend"]   = (df["day_of_week"] >= 5).astype(int)
df["is_off_hours"] = ((df["hour"] < 8) | (df["hour"] >= 20)).astype(int)

In [8]:
# -- Alert velocity: alerts from same src_ip in a 15-minute window --
# Burst activity (e.g. a real port scan or brute force) produces many alerts
# quickly. FP sources like patch management fire at a steady low rate.
# Data dictionary notes: 15-min window drops FPR from 65% → <5%.

# Sort by src_ip then time so groupby+rolling works correctly

#This method is better than previous since its time complexityis O(N logN)

df = df.sort_values(["src_ip", "ts_epoch"]).reset_index(drop=True)

# Set timestamp as index — required for time-based rolling
df.index = pd.to_datetime(df["ts_epoch"], unit="s")

# For each src_ip group, count alerts in a rolling 15-min window around each row.
# rolling("30min") looks 30 min back; we want ±15 min so we use 30 min back
# then shift — simpler: just count 15 min back (forward-only window is fine
# because burst alerts arrive in sequence anyway).
df["alert_velocity_15m"] = (
    df.groupby("src_ip")["ts_epoch"]
    .rolling("15min")
    .count()
    .values
)

# Restore the default integer index
df = df.reset_index(drop=True)

In [9]:
# -- Source IP behavioral aggregates --
# Computed globally across all alerts (no label involvement).
# An IP hitting many unique destinations or ports behaves like a scanner;
# an IP with very few connections that fire a critical alert looks targeted.
src_stats = (
    df.groupby("src_ip")
    .agg(
        src_total_alerts   = ("alert_id",  "count"),
        src_unique_dst_ips = ("dst_ip",    "nunique"),
        src_unique_ports   = ("dst_port",  "nunique"),
    )
    .reset_index()
)
df = df.merge(src_stats, on="src_ip", how="left")
 

In [10]:
# -- Severity as ordinal --
# Suricata assigns severity; the model learns how reliable it actually is.
severity_map = {"low": 1, "medium": 2, "high": 3, "critical": 4}
df["severity_num"] = df["severity"].map(severity_map).fillna(2).astype(int)

In [11]:
# -- Categorical encoding --
# LabelEncoder is adequate here; RF splits on integer codes just fine.
# signature_category (PORT_SCAN, C2, LATERAL…) is a strong behavioral signal.
# mitre_tactic encodes the attacker's goal — different tactics have different FP rates.
for col in ["signature_category", "protocol", "mitre_tactic"]:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col].fillna("UNKNOWN").astype(str))

In [12]:
# -- Confidence score --
# The rule engine's own estimate. Not ground truth — it is derived from rule
# metadata, not from is_true_positive. Safe to include.
df["confidence_score"] = df["confidence_score"].fillna(0.5)

In [13]:
# -- Payload size --
# FP alert sources (backup jobs, scanners) often have characteristic packet sizes.
df["raw_payload_bytes"] = df["raw_payload_bytes"].fillna(0)

In [14]:
# -- Tactic seen in confirmed incident (ticket enrichment, not label-derived) --
# Weak signal: if this alert's MITRE tactic appeared in a real confirmed attack
# incident, the model should weight it accordingly.
df["tactic_in_confirmed"] = df["mitre_tactic"].apply(
    lambda t: 1 if str(t).strip() in confirmed_tactics else 0
)
 
print(f"    Feature engineering complete.")

    Feature engineering complete.


In [15]:
# ── 4. TRAIN / TEST SPLIT ────────────────────────────────────────────────────
# Split BEFORE any label-dependent computation (there is none here, but good habit).
print("\n[4] Splitting train / test (80 / 20, stratified)...")
 
FEATURE_COLS = [
    # Time behavior
    "hour", "day_of_week", "is_weekend", "is_off_hours",
    # Burst / velocity
    "alert_velocity_15m",
    # Source IP behavior across all its alerts
    "src_total_alerts", "src_unique_dst_ips", "src_unique_ports",
    # Alert metadata
    "severity_num", "confidence_score", "raw_payload_bytes",
    # Encoded categoricals
    "signature_category_enc", "protocol_enc", "mitre_tactic_enc",
    # Ticket enrichment
    "tactic_in_confirmed",
]
 
X = df[FEATURE_COLS].fillna(0).values
y = df["is_true_positive"].astype(int).values
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"    Train: {len(X_train):,}  |  Test: {len(X_test):,}")
 
 


[4] Splitting train / test (80 / 20, stratified)...
    Train: 120,000  |  Test: 30,000


In [16]:
# ── 5. TRAIN ─────────────────────────────────────────────────────────────────
print("\n[5] Training Random Forest...")
 
model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",    # corrects for 65/35 imbalance automatically
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
model.fit(X_train, y_train)


[5] Training Random Forest...


,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [17]:
# ── 6. EVALUATE ──────────────────────────────────────────────────────────────
print("\n[6] Evaluation on held-out test set")
print("-" * 60)
 
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)
 
print(classification_report(y_test, y_pred, target_names=["False Positive", "True Positive"]))
print(f"    ROC-AUC : {roc_auc_score(y_test, y_prob):.4f}")
 


[6] Evaluation on held-out test set
------------------------------------------------------------
                precision    recall  f1-score   support

False Positive       0.84      0.96      0.90     19500
 True Positive       0.90      0.66      0.76     10500

      accuracy                           0.85     30000
     macro avg       0.87      0.81      0.83     30000
  weighted avg       0.86      0.85      0.85     30000

    ROC-AUC : 0.9235


In [18]:
# False Positive Rate = FP / (FP + TN)
# i.e. of all real negatives, how many did we incorrectly flag?
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
model_fpr    = fp / (fp + tn)
baseline_fpr = 1 - alerts["is_true_positive"].mean()   # raw alert stream FPR
 
print(f"\n    Baseline FP rate : {baseline_fpr*100:.1f}%  (all alerts treated as TP)")
print(f"    Model FP rate    : {model_fpr*100:.1f}%  (on test set)")
print(f"    FP reduction     : {(baseline_fpr - model_fpr)*100:.1f} percentage points")
 


    Baseline FP rate : 65.0%  (all alerts treated as TP)
    Model FP rate    : 3.9%  (on test set)
    FP reduction     : 61.1 percentage points


In [19]:
# ── 7. FEATURE IMPORTANCE ────────────────────────────────────────────────────
print("\n[7] What the model learned (feature importances):")
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS)
for feat, imp in importances.sort_values(ascending=False).head(8).items():
    bar = "█" * int(imp * 60)
    print(f"    {feat:<30} {bar}  {imp:.3f}")


[7] What the model learned (feature importances):
    confidence_score               ██████████████████████  0.374
    alert_velocity_15m             ████████  0.136
    hour                           ██████  0.100
    raw_payload_bytes              █████  0.094
    src_total_alerts               ████  0.069
    src_unique_dst_ips             ███  0.058
    day_of_week                    ███  0.052
    src_unique_ports               ██  0.047


In [20]:
# ── 8. SCORE ALL ALERTS & SAVE ───────────────────────────────────────────────
print("\n[8] Scoring all alerts and saving results...")
 
df["triage_score"]    = model.predict_proba(df[FEATURE_COLS].fillna(0).values)[:, 1]
df["predicted_tp"]    = (df["triage_score"] >= 0.5).astype(int)
df["predicted_label"] = df["predicted_tp"].map({1: "TRUE_POSITIVE", 0: "FALSE_POSITIVE"})
 
out_cols = [
    "alert_id", "timestamp", "severity", "signature_category",
    "mitre_tactic", "mitre_technique",
    "src_ip", "dst_ip", "dst_port", "protocol",
    "confidence_score", "alert_velocity_15m",
    "triage_score", "predicted_label",
    "is_true_positive",    # ground truth included for hackathon evaluation
]


[8] Scoring all alerts and saving results...


In [21]:
output = df[out_cols].sort_values("triage_score", ascending=False)
output.to_csv(OUTPUT_CSV, index=False)
 
total      = len(df)
suppressed = (df["predicted_tp"] == 0).sum()
print(f"\n    Total alerts     : {total:,}")
print(f"    Flagged as TP    : {(df['predicted_tp']==1).sum():,}")
print(f"    Suppressed as FP : {suppressed:,}  ({suppressed/total*100:.1f}% noise reduction)")
 
print(f"\n    Top 10 highest-confidence true threats:")
print(f"    {'Alert ID':<25} {'Score':>6}  {'Category':<20}  {'Tactic':<25}  {'Velocity':>8}")
print("    " + "-" * 88)
for _, r in output[output["predicted_label"] == "TRUE_POSITIVE"].head(10).iterrows():
    print(
        f"    {str(r['alert_id']):<25} {r['triage_score']:>6.3f}"
        f"  {str(r['signature_category']):<20}"
        f"  {str(r['mitre_tactic']):<25}"
        f"  {int(r['alert_velocity_15m']):>8}"
    )



    Total alerts     : 150,000
    Flagged as TP    : 49,741
    Suppressed as FP : 100,259  (66.8% noise reduction)

    Top 10 highest-confidence true threats:
    Alert ID                   Score  Category              Tactic                     Velocity
    ----------------------------------------------------------------------------------------
    IDS-20260128-EC2FC4AC      1.000  ATM_ATTACK            TA0040-Impact                     1
    IDS-20260619-F558D894      1.000  ATM_ATTACK            TA0040-Impact                     1
    IDS-20260624-A0DD787E      1.000  ATM_ATTACK            TA0040-Impact                     1
    IDS-20260625-347296A4      1.000  PRIV_ESC              TA0006-Credential Access          1
    IDS-20251221-6B1B7EB2      1.000  ATM_ATTACK            TA0040-Impact                     1
    IDS-20250906-C45AF6C7      1.000  ATM_ATTACK            TA0040-Impact                     1
    IDS-20250418-E45C667D      1.000  PRIV_ESC              TA0006-Crede

In [22]:
print(f"\n[9] Results saved → {OUTPUT_CSV}")
print("=" * 60)


[9] Results saved → alert_triage_results.csv
